# 🐦 Cantos de Aves — TinyML · Entrenamiento (Metodología KWS)

Pipeline idéntico al Ejercicio KWS (helios/tennis/poker), ahora con **6 clases de aves**:
1. Cargar audios → recorte de 1 s con más energía
2. **MFCC a mano en numpy** (ventana 25 ms, salto 10 ms, 40 filtros mel, log antes de la DCT, 13 coeficientes)
3. Aplanado a 1.274 features
4. **Experimento A** (split aleatorio — con trampa de fuga) vs **Experimento B** (leave-one-session-out — honesto)
5. Export del modelo para la app (`modelo_aves.zip`) — tfjs + labels

**Antes de correr:** sube tu dataset (zip de Edge Impulse renombrado con `_S<n>_`, ver `edge-impulse/README.md`) y pon su URL en la celda 2, o súbelo a mano con el panel de archivos.

In [ ]:
import zipfile, io, os, glob, wave
import numpy as np
import matplotlib.pyplot as plt

SR = 16000
VENTANA = 25e-3          # 25 ms (400 muestras @16k)
SALTO = 10e-3            # 10 ms (160 muestras @16k)
N_MEL = 40
N_COEFS = 13
N_MUESTRAS_UTILES = 16000          # ventana de análisis: 1 s
FRAMES = 98                        # 25 ms/10 ms sobre 1 s → 98 frames
N_FEATURES = FRAMES * N_COEFS      # 1.274  (igual que el KWS)

CLASES = ['garrapatero-comun','pellar-comun','pigua','bichofue-griton','azulejo-comun','lora-andina']
print('CLASES:', CLASES, '| features por ejemplo:', N_FEATURES)

## 1 · Datos
Opción A: URL directa del zip (Drive con permiso 'cualquiera con el enlace' → usar el enlace `uc?export=download&id=...`).  
Opción B: arrastrar el zip al panel de archivos (icono 📁 a la izquierda) y dejar `DATA_URL = None`.

In [ ]:
DATA_URL = None   # p. ej. 'https://drive.google.com/uc?export=download&id=XXXXXXXX'

if DATA_URL:
    !wget -q -O dataset_aves.zip "{DATA_URL}"
    with zipfile.ZipFile('dataset_aves.zip') as z: z.extractall('data')

# Buscar todas las wav bajo /data
rutas = sorted(glob.glob('/content/data/**/*.wav', recursive=True))
print('audios encontrados:', len(rutas))
print(rutas[:3] if rutas else '⚠️ ¡Sin audios! Revisa el zip o la carpeta data/')

## 2 · Cargar + recorte por energía
Audios de 3–5 s → nos quedamos con la **ventana de 1 s con más energía** (los cantos no ocupan todo el clip). Remuestreo por decimación lineal si hace falta (Edge Impulse exporta a 16 kHz mono, igual que el KWS).

In [ ]:
def cargar_wav(ruta):
    w = wave.open(ruta, 'rb')
    sr = w.getframerate(); n = w.getnframes(); ch = w.getnchannels()
    x = np.frombuffer(w.readframes(n), dtype=np.int16).astype(np.float64)
    w.close()
    if ch > 1: x = x.reshape(-1, ch).mean(axis=1)
    if sr != SR:  # remuestreo lineal simple
        x = np.interp(np.linspace(0, len(x)-1, int(len(x)*SR/sr)), np.arange(len(x)), x)
    return x / (np.abs(x).max() + 1e-9)

def recorte_energia(x, n=N_MUESTRAS_UTILES):
    if len(x) <= n: return np.pad(x, (0, n-len(x)))
    paso = 1000
    energias = [np.sum(x[i:i+n]**2) for i in range(0, len(x)-n, paso)]
    i = int(np.argmax(energias)) * paso
    return x[i:i+n]

def clase_desde_ruta(ruta):
    for c in CLASES:
        if c in ruta: return c
    return None

def sesion_desde_ruta(ruta):
    import re
    m = re.search(r'_S(\d+)_', ruta)
    return int(m.group(1)) if m else None

X, y, sesiones, rutas_ok = [], [], [], []
for r in rutas:
    c = clase_desde_ruta(r)
    if c is None or sesion_desde_ruta(r) is None:
        print('⏭️ sin clase/sesión:', r); continue
    X.append(recorte_energia(cargar_wav(r)))
    y.append(CLASES.index(c)); sesiones.append(sesion_desde_ruta(r)); rutas_ok.append(r)
X = np.array(X).astype(np.float32); y = np.array(y); sesiones = np.array(sesiones)
print('✅ cargados:', X.shape, '| por clase:', {c:int((y==i).sum()) for i,c in enumerate(CLASES)})
print('sesiones:', sorted(set(sesiones.tolist())))

In [ ]:
# Exploratoria: 6 cantos, uno por clase
fig, axs = plt.subplots(6, 1, figsize=(9, 9))
for i, c in enumerate(CLASES):
    idx = np.where(y == i)[0][0]
    axs[i].plot(X[idx], lw=.4); axs[i].set_title(c, fontsize=9, loc='left'); axs[i].set_yticks([])
plt.tight_layout(); plt.savefig('exploratoria_aves.png', dpi=110)

## 3 · MFCC a mano en numpy (lo mismo del KWS)
Pre-énfasis → frames (25/10 ms) → ventana de Hann → FFT → filtrobank mel (40) → **log antes de la DCT** → DCT → 13 coeficientes.

In [ ]:
def hz_a_mel(f):  return 2595 * np.log10(1 + f / 700)
def mel_a_hz(m):  return 700 * (10**(m / 2595) - 1)

def filtrobank_mel(sr=SR, n_fft=512, n_mel=N_MEL, fmin=50, fmax=8000):
    mel_pts = np.linspace(hz_a_mel(fmin), hz_a_mel(fmax), n_mel + 2)
    bins = np.floor((n_fft + 1) * mel_a_hz(mel_pts) / sr).astype(int)
    fb = np.zeros((n_mel, n_fft // 2 + 1))
    for m in range(1, n_mel + 1):
        izq, ctr, der = bins[m-1], bins[m], bins[m+1]
        if ctr == izq: ctr += 1
        if der == ctr: der += 1
        fb[m-1, izq:ctr] = (np.arange(izq, ctr) - izq) / (ctr - izq)
        fb[m-1, ctr:der] = 1 - (np.arange(ctr, der) - ctr) / (der - ctr)
    return fb

FB = filtrobank_mel()
DCT_MAT = np.fromfunction(lambda k, n: np.cos(np.pi * k * (2*n+1) / (2*N_MEL)),
                          (N_COEFS, N_MEL)) * np.sqrt(2/N_MEL)
DCT_MAT[0] *= np.sqrt(1/2)

def mfcc(x, sr=SR):
    x = np.append(x[0], x[1:] - 0.97 * x[:-1])                # pre-énfasis
    L, n_fft = len(x), 512
    frame, hop = int(VENTANA*sr), int(SALTO*sr)
    n_frames = 1 + (L - frame) // hop
    hann = 0.5 - 0.5*np.cos(np.linspace(0, 2*np.pi, frame, endpoint=False))
    frames = np.lib.stride_tricks.sliding_window_view(x, frame)[::hop][:n_frames] * hann
    # STFT
    mag = np.abs(np.fft.rfft(frames, n_fft, axis=1))**2
    mel = np.log(mag @ FB.T + 1e-10)                          # log ANTES de la DCT
    return (mel @ DCT_MAT.T)[:, :N_COEFS]                     # (n_frames, 13)

def features(x):
    m = mfcc(recorte_energia(x))
    if m.shape[0] < FRAMES: m = np.vstack([m, np.zeros((FRAMES - m.shape[0], N_COEFS))])
    return m[:FRAMES].flatten()                                # 1.274

F = np.stack([features(x) for x in X]).astype(np.float32)
print('✅ features:', F.shape)
plt.figure(figsize=(5,4)); plt.imshow(mfcc(X[0]).T, aspect='auto', origin='lower');
plt.title('MFCC ejemplo: ' + CLASES[y[0]], fontsize=9); plt.colorbar()

## 4 · Modelo — MLP en puro numpy (mismo estilo del KWS)
1274 → 64 (ReLU) → 6 (softmax) · cross-entropy · mini-batches.

In [ ]:
def inicializar(n_in=N_FEATURES, oc=64, n_out=len(CLASES), semilla=0):
    rng = np.random.default_rng(semilla)
    W1 = rng.normal(0, np.sqrt(2/n_in), (n_in, oc)).astype(np.float32)
    b1 = np.zeros(oc, np.float32)
    W2 = rng.normal(0, np.sqrt(2/oc), (oc, n_out)).astype(np.float32)
    b2 = np.zeros(n_out, np.float32)
    return {'W1':W1,'b1':b1,'W2':W2,'b2':b2}

def forward(m, X, entrenar=False):
    z1 = X @ m['W1'] + m['b1']; a1 = np.maximum(z1, 0)
    z2 = a1 @ m['W2'] + m['b2']
    z2 -= z2.max(axis=1, keepdims=True)
    p = np.exp(z2); p /= p.sum(axis=1, keepdims=True)
    cache = (X, z1, a1, p) if entrenar else None
    return p, cache

def entrenar(m, Xtr, ytr, epocas=60, bs=16, lr=1e-2, semilla=0):
    rng = np.random.default_rng(semilla); n = len(Xtr)
    hist = []
    for ep in range(epocas):
        idx = rng.permutation(n)
        for i in range(0, n, bs):
            b = idx[i:i+bs]
            Xb, yb = Xtr[b], ytr[b]
            p, (Xi, z1, a1, _) = forward(m, Xb, entrenar=True)
            dz2 = p.copy(); dz2[np.arange(len(b)), yb] -= 1
            dz2 /= len(b)
            dW2 = a1.T @ dz2; db2 = dz2.sum(0)
            da1 = dz2 @ m['W2'].T
            dz1 = da1 * (z1 > 0)
            dW1 = Xi.T @ dz1; db1 = dz1.sum(0)
            m['W2'] -= lr*dW2; m['b2'] -= lr*db2
            m['W1'] -= lr*dW1; m['b1'] -= lr*db1
        # pérdida y acc de train
        ptr, _ = forward(m, Xtr)
        plog = -np.log(ptr[np.arange(n), ytr] + 1e-9).mean()
        hist.append((ep, float(plog)))
    return hist

def exactitud(m, X, y):
    p, _ = forward(m, X)
    return float((p.argmax(1) == y).mean()), p

## 5 · Experimento A — split aleatorio (con la trampa)
Mezcla **todo** al azar y separa 80/20. Clips vecinos de una misma sesión caen en ambos lados: la red aprende el "brillo" de la sesión, no el canto. Espera un número **demasiado bonito**.

In [ ]:
def media_desv(F):
    mu, sd = F.mean(0), F.std(0) + 1e-6
    return mu, sd

rng = np.random.default_rng(0)
idx = rng.permutation(len(F)); n_tr = int(.8*len(F))
trA, teA = idx[:n_tr], idx[n_tr:]
mu, sd = media_desv(F[trA])
mA = inicializar(semilla=1)
histA = entrenar(mA, (F[trA]-mu)/sd, y[trA])
accA, _ = exactitud(mA, (F[teA]-mu)/sd, y[teA])
print(f'🅰️ Experimento A (split aleatorio): exactitud = {accA*100:.1f}%  ← ¡demasiado bonita?')
# ¿de qué sesiones viene el test? → fuga
ses_tr, ses_te = set(sesiones[trA].tolist()), set(sesiones[teA].tolist())
print(f'   sesiones compartidas entre train y test: {sorted(ses_tr & ses_te)}  ← fuga de datos si hay cruce')

## 6 · Experimento B — leave-one-session-out (honesto)
Entrena con **todas** las sesiones menos una, prueba con la sesión dejada fuera, y rota. El promedio es la métrica real.

In [ ]:
sesiones_unicas = sorted(set(sesiones.tolist()))
accs, matrices = [], np.zeros((len(CLASES), len(CLASES)), int)
for s_out in sesiones_unicas:
    tr, te = sesiones != s_out, sesiones == s_out
    mu, sd = media_desv(F[tr])
    mB = inicializar(semilla=1)
    entrenar(mB, (F[tr]-mu)/sd, y[tr])
    acc, p = exactitud(mB, (F[te]-mu)/sd, y[te])
    accs.append(acc)
    pred = p.argmax(1)
    for t, pr in zip(y[te], pred): matrices[t, pr] += 1
    print(f'  deja fuera sesión S{s_out}: prueba sobre {int(te.sum())} clips → {acc*100:.1f}%')
accB = float(np.mean(accs))
print(f'\n🅱️ Experimento B (leave-one-session-out): {accB*100:.1f}%  ← la métrica honesta')
print(f'   caída A→B: {(accA-accB)*100:.1f} puntos = la fuga de sesión del Experimento A')

fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(matrices); ax.set_xticks(range(6), CLASES, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(6), CLASES, fontsize=8)
for i in range(6):
    for j in range(6): ax.text(j, i, matrices[i,j], ha='center', va='center', fontsize=9)
plt.tight_layout(); plt.savefig('matriz_confusion_B.png', dpi=110)

## 7 · Export para la app (tfjs + labels)
Pasamos el numpy a Keras/TF con los **mismos pesos**, y exportamos para el navegador. Descarga `modelo_aves.zip` y descomprímelo en `app/modelo/`.

In [ ]:
# Entrenamos el modelo final con TODOS los datos (sesiones combinadas) ya validado por B
mu_full, sd_full = media_desv(F)
mF = inicializar(semilla=1)
entrenar(mF, (F-mu_full)/sd_full, y, epocas=80)
accF, _ = exactitud(mF, (F-mu_full)/sd_full, y)
print(f'modelo final sobre todo el train: {accF*100:.1f}%')
np.savez('params_final.npz', W1=mF['W1'], b1=mF['b1'], W2=mF['W2'], b2=mF['b2'], mu=mu_full, sd=sd_full)

In [ ]:
!pip -q install tensorflow tensorflowjs
import tensorflow as tf, tensorflowjs as tfjs, json, shutil

p = np.load('params_final.npz')
k = tf.keras.Sequential([
    tf.keras.layers.Input((N_FEATURES,)),
    tf.keras.layers.Dense(64, activation='relu', weights=[p['W1'], p['b1']]),
    tf.keras.layers.Dense(len(CLASES), activation='softmax', weights=[p['W2'], p['b2']]),
])
k.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
preds_k = k.predict((F-mu_full)/sd_full, verbose=0)
print(' sanity keras vs numpy:', float(np.abs(preds_k - forward(mF,(F-mu_full)/sd_full)[0]).max()))

shutil.rmtree('modelo_export', ignore_errors=True)
tfjs.converters.save_keras_model(k, 'modelo_export')
with open('modelo_export/labels.json', 'w') as f:
    json.dump({'clases': CLASES, 'features': N_FEATURES, 'frames': FRAMES, 'coefs': N_COEFS,
               'sr': SR, 'mu': mu_full.tolist(), 'sd': sd_full.tolist()}, f)
!zip -qr modelo_aves.zip modelo_export
from google.colab import files
files.download('modelo_aves.zip')
print('✅ descarga modelo_aves.zip → app/modelo/')